# SadTalker Quick Demo (Modern Colab)

This notebook is updated for modern Colab runtimes, current CUDA-enabled PyTorch, and current pip resolver behavior.


## 1) Clone repository


In [ ]:

# =========================
# Cell 1: Clone repository
# =========================
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Inialpha/SadTalker.git"
BRANCH = "python312-modernization"
REPO_DIR = Path("/content/SadTalker")

if not REPO_DIR.exists():
    print(f"Cloning {REPO_URL} (branch: {BRANCH}) -> {REPO_DIR}")
    subprocess.run(
        [
            "git",
            "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )
else:
    print(f"Repository already exists at {REPO_DIR}")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)

print("Python:", sys.version)

current_branch = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "branch", "--show-current"],
    text=True,
).strip()
print("Current branch:", current_branch)

Cloning https://github.com/Inialpha/SadTalker.git (branch: python312-modernization) -> /content/SadTalker
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Current branch: python312-modernization


In [ ]:

# ====================================
# Cell 2: System upgrade / system deps
# ====================================
import shutil
import subprocess

if shutil.which("ffmpeg") is None:
    print("Installing ffmpeg...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)
else:
    print("ffmpeg already installed")

ffmpeg already installed


In [ ]:

# ============================================
# Cell 3: Python package bootstrap and install
# ============================================
from pathlib import Path
import importlib
import subprocess
import sys

REPO_DIR = Path("/content/SadTalker")
REQ_FILE = REPO_DIR / "requirements.txt"

if not REQ_FILE.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQ_FILE}")

print("Upgrading pip, setuptools, and wheel...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)

# Packages that should generally be left alone if Colab already provides them.
protected = {
    "numpy",
    "scipy",
    "torch",
    "torchvision",
    "torchaudio",
    "opencv-python",
    "opencv-python-headless",
}

# Requirement name -> import name mapping.
module_map = {
    "face-alignment": "face_alignment",
    "imageio-ffmpeg": "imageio_ffmpeg",
    "scikit-image": "skimage",
    "pyyaml": "yaml",
    "opencv-python": "cv2",
    "opencv-python-headless": "cv2",
    "pillow": "PIL",
}

def parse_requirement_name(req: str) -> str:
    req = req.strip()
    for sep in ("==", ">=", "<=", "~=", "!="):
        if sep in req:
            req = req.split(sep, 1)[0]
    req = req.split("[", 1)[0]
    return req.strip()

def import_name_for(requirement_name: str) -> str:
    key = requirement_name.lower()
    if key in module_map:
        return module_map[key]
    return key.replace("-", "_")

requirements = []

for raw_line in REQ_FILE.read_text().splitlines():
    line = raw_line.strip()
    if not line or line.startswith("#"):
        continue

    line = line.split("#", 1)[0].strip()
    if not line:
        continue

    req_name = parse_requirement_name(line).lower()

    if req_name in protected:
        print(f"Skipping protected package: {req_name}")
        continue

    requirements.append(line)

for requirement in requirements:
    req_name = parse_requirement_name(requirement).lower()
    module_name = import_name_for(req_name)

    try:
        importlib.import_module(module_name)
        print(f"✓ {module_name} already installed")
    except Exception:
        print(f"Installing {requirement}")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", requirement],
            check=True,
        )

print("Dependency installation complete.")

Upgrading pip, setuptools, and wheel...
Skipping protected package: numpy
Skipping protected package: scipy
Installing face-alignment==1.4.1
✓ imageio already installed
✓ imageio_ffmpeg already installed
✓ librosa already installed
✓ numba already installed
Installing resampy==0.4.3


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


✓ pydub already installed
Installing kornia==0.7.3
✓ tqdm already installed
Installing yacs==0.1.8
✓ yaml already installed
✓ joblib already installed
✓ skimage already installed
Installing basicsr==1.4.2
Installing facexlib==0.3.0
✓ gradio already installed
Installing gfpgan
Installing av
✓ safetensors already installed
✓ huggingface_hub already installed
Dependency installation complete.


## 2) Install dependencies


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3) Download model checkpoints


In [9]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/SadTalker")
print("Downloading checkpoints with scripts/download_models.sh ...")
subprocess.run(["bash", str(REPO_DIR / "scripts" / "download_models.sh")], check=True, cwd=REPO_DIR)
print("Download complete")

FileNotFoundError: [Errno 2] No such file or directory: PosixPath('/content/SadTalker')

## 4) Verify checkpoints


In [ ]:
from pathlib import Path

REPO_DIR = Path("/content/SadTalker")
ckpt = REPO_DIR / "checkpoints"

required = [
    "SadTalker_V0.0.2_256.safetensors",
    "SadTalker_V0.0.2_512.safetensors",
    "mapping_00109-model.pth.tar",
    "mapping_00229-model.pth.tar",
]

missing = [name for name in required if not (ckpt / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing checkpoints: {missing}")

print("Checkpoint verification passed")
for name in required:
    path = ckpt / name
    print(f"- {name}: {path.stat().st_size / (1024**2):.1f} MB")

Checkpoint verification passed
- SadTalker_V0.0.2_256.safetensors: 691.5 MB
- SadTalker_V0.0.2_512.safetensors: 691.5 MB
- mapping_00109-model.pth.tar: 148.6 MB
- mapping_00229-model.pth.tar: 148.3 MB


## 5) Import libraries and verify runtime


In [ ]:
import platform
import sys

import torch
import torchvision
import numpy as np
import scipy
import imageio
import librosa
import gradio as gr

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("imageio:", imageio.__version__)
print("librosa:", librosa.__version__)
print("gradio:", gr.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Torch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA available: True
CUDA device: Tesla T4
NumPy: 2.0.2
SciPy: 1.16.3
imageio: 2.37.3
librosa: 0.11.0
gradio: 6.20.0


In [2]:
%xmode verbose

Exception reporting mode: Verbose


In [8]:
%cd SadTalker

[Errno 2] No such file or directory: 'SadTalker'
/content


In [1]:
!git pull

fatal: not a git repository (or any of the parent directories): .git


## 6) Load models


In [11]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/SadTalker")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.utils.init_path import init_path
from src.utils.preprocess import CropAndExtract
from src.test_audio2coeff import Audio2Coeff
from src.facerender.animate import AnimateFromCoeff

checkpoint_dir = REPO_DIR / "checkpoints"
config_dir = REPO_DIR / "src" / "config"
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"

paths = init_path(str(checkpoint_dir), str(config_dir), 256, False, "crop")
print("Resolved model paths:")
for k, v in paths.items():
    print(f"- {k}: {v}")

preprocess_model = CropAndExtract(paths, device)
audio_to_coeff = Audio2Coeff(paths, device)
animate_from_coeff = AnimateFromCoeff(paths, device)
print("Model components loaded on", device)

AttributeError: module 'numpy' has no attribute 'VisibleDeprecationWarning'

## 7) Run inference


In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/SadTalker")
image_path = REPO_DIR / "examples" / "source_image" / "full_body_1.png"
audio_path = REPO_DIR / "examples" / "driven_audio" / "bus_chinese.wav"
result_dir = REPO_DIR / "results" / "quick_demo"
result_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    "python", "inference.py",
    "--driven_audio", str(audio_path),
    "--source_image", str(image_path),
    "--result_dir", str(result_dir),
    "--still",
    "--preprocess", "full",
    "--enhancer", "gfpgan",
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=REPO_DIR)
print("Inference completed")


## 8) Display results


In [ ]:
from pathlib import Path
from IPython.display import Video, display

REPO_DIR = Path("/content/SadTalker")
videos = sorted((REPO_DIR / "results" / "quick_demo").glob("*.mp4"), key=lambda p: p.stat().st_mtime)
if not videos:
    raise FileNotFoundError("No output video found in /content/SadTalker/results/quick_demo")

latest = videos[-1]
print("Latest output:", latest)
display(Video(str(latest), embed=True, width=512))


## 9) Optional Gradio interface


In [ ]:
# Optional: launch the interactive Gradio interface.
# Stop the cell to close the app.

from app_sadtalker import sadtalker_demo

demo = sadtalker_demo(checkpoint_path='checkpoints', config_path='src/config')
demo.queue()
demo.launch(share=False)
